# In-context learning, from zero

A learning notebook: build up in-context learning piece by piece, watching each idea land before adding the next.

In [2]:
import torch
import numpy as np

torch.manual_seed(42)
print("Torch device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Torch device: cuda


## Dataset — matching the paper, not TinyStories

Olsson et al. 2022 trained their small models on an earlier version of the corpus described in
Askell et al. 2021 (arXiv:2112.00861): **filtered Common Crawl + internet books + ~10% Python
code**, plus several smaller distributions. Askell et al.'s exact corpus was never released.

We use `monology/pile-uncopyrighted` instead: The Pile with the five subsets under copyright
takedown removed (Books3, BookCorpus2, OpenSubtitles, YTSubtitles, OWT2). What remains — Pile-CC
(filtered Common Crawl), Gutenberg (books), GitHub (code, ~7-8% of tokens in the original Pile mix
— close to the paper's ~10%), plus the same long tail of smaller distributions Askell et al.
describe — is the closest reliably-downloadable public match. It's also what Pythia, this
project's other reference model family, was trained on.

**Why this matters for what we're building:** the earlier `stage2c_induction_tinystories.ipynb`
notebook, trained on TinyStories, never reproduced the paper's sharp induction-score phase
transition — the rise was gradual instead. TinyStories has no code and only short documents;
code's near-verbatim repetition may be exactly what sharpens that signal in the paper. Training on
a Pile-like mix lets us actually test that, instead of assuming it.

In [ ]:
DATASET_NAME = "monology/pile-uncopyrighted"  # near-identical to Olsson et al.'s corpus
CHAR_BUDGET = 200_000_000  # same order of magnitude as the stage2c TinyStories run
CACHE_DIR = "checkpoints/icl_pile"
CORPUS_CACHE = f"{CACHE_DIR}/corpus.txt"

VOCAB_SIZE = 2048  # our own byte-level BPE; chosen by a fertility sweep (see below)
TOKENIZER_CACHE = f"{CACHE_DIR}/tokenizer.json"
TOKENS_CACHE = f"{CACHE_DIR}/tokens_v{VOCAB_SIZE}.npy"

print(
    f"[config] dataset={DATASET_NAME!r} char_budget={CHAR_BUDGET:,} cache={CORPUS_CACHE!r}\n"
    f"[config] vocab_size={VOCAB_SIZE:,} tokenizer={TOKENIZER_CACHE!r} tokens={TOKENS_CACHE!r}"
)

## Smoke test — does this actually stream, and is it what we expect?

Before pulling `CHAR_BUDGET` characters, pull just 5 rows and look at them. Each row is
`{"text": ..., "meta": {"pile_set_name": ...}}` — `pile_set_name` tells us which of the Pile's
original components a row came from, so we can sanity-check the mix (Common Crawl, code, books,
...) before committing to a full pull.

In [4]:
from datasets import load_dataset

_stream = load_dataset(DATASET_NAME, split="train", streaming=True)
_sample = [row for _, row in zip(range(5), _stream)]

for i, row in enumerate(_sample):
    source = row["meta"]["pile_set_name"]
    preview = row["text"][:100].replace("\n", " ")
    print(f"[{i}] source={source!r}")
    print(f"    {preview!r}")

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

[0] source='Pile-CC'
    'It is done, and submitted. You can play “Survival of the Tastiest” on Android, and on the web. Playi'
[1] source='Github'
    '<?xml version="1.0" encoding="UTF-8"?>\r <segment>\r     <name>PD1</name>\r     <description>Patient Ad'
[2] source='Pile-CC'
    'Topic: reinvent midnight madness  Amazon announced a new service at the AWS re:Invent Midnight Madne'
[3] source='Pile-CC'
    'About Grand Slam Fishing Charters  As a family owned business we know how important it is that your '
[4] source='StackExchange'
    "Q:  Why was Mundungus banned from the Hog's Head?  In Order of the Phoenix while the trio were in th"


## Model architecture

Two-layer, attention-only is the tractable case Olsson et al. can write the exact circuit for —
but the paper's own text only commits to that qualitative shape ("2-layer attention-only");
Anthropic never released the actual weights. Several community-trained models fill that gap in
`transformer_lens`, and their configs are **not interchangeable**, so it's worth being precise
about which one we mean:

- `transformer_lens`'s `attn-only-2l` alias resolves to `NeelNanda/Attn_Only_2L512W_C4_Code`:
  `d_model=512`, standard positional embeddings, trained on C4+Code.
- `ArthurConmy/redwood_attn_2l` is a separate, smaller model: `d_model=256`, shortformer
  positional embeddings, `d_vocab=50259`.

We adopt **`redwood_attn_2l`'s architecture numbers** below (verified against its real
`config.json` on HuggingFace, not from memory) as **our own training config** — training this
architecture from scratch on the Pile-uncopyrighted corpus configured above, not loading
`redwood_attn_2l`'s pretrained weights.

`d_vocab` is the one number we deliberately *don't* borrow from `redwood_attn_2l`: it's a
property of its tokenizer, not the architecture, and that tokenizer was fit to whatever corpus
Arthur Conmy trained on — not ours. Dataset preparation (below) trains our own byte-level BPE
tokenizer on the Pile-uncopyrighted corpus instead, sized to `VOCAB_SIZE` — small enough that the
embedding/unembedding matrices stay proportionate to this model's width (`d_model=256`) rather
than dwarfing it the way a 50k-vocab GPT-2 tokenizer would.

| Setting | Value | Note |
|---|---|---|
| `n_layers` | 2 | |
| `d_model` | 256 | |
| `n_heads` | 8 | |
| `d_head` | 32 | = d_model / n_heads |
| `n_ctx` | 2048 | context length |
| `d_vocab` | `VOCAB_SIZE` (2048) | our own tokenizer, not redwood's — see above |
| `attn_only` | `True` | no MLP blocks — layers compose purely through attention |
| `normalization_type` | `"LN"` | standard LayerNorm |
| `positional_embedding_type` | `"shortformer"` | positional info injected into Q/K, not added to the residual stream at layer 0 |
| Tokenizer | byte-level BPE | trained fresh on our corpus in dataset preparation, GPT-2's algorithm not its vocabulary |

**Why this architecture matters for induction:** exactly 2 layers forces layer-0 heads into the
"previous-token head" role (there's nothing earlier to compose from) and layer-1 heads into the
"induction head" role via K-composition — the same depth argument the paper itself makes for why
one layer can't do this.

In [ ]:
MODEL_CONFIG = dict(
    n_layers=2,
    d_model=256,
    n_heads=8,
    d_head=32,
    n_ctx=2048,
    d_vocab=VOCAB_SIZE,  # our own tokenizer (dataset prep below), not redwood_attn_2l's 50259
    attn_only=True,
    normalization_type="LN",
    positional_embedding_type="shortformer",
)

n_params_estimate = 4 * MODEL_CONFIG["d_model"] ** 2 * MODEL_CONFIG["n_layers"]  # attn-only, QKVO per layer
embed_params_estimate = 2 * MODEL_CONFIG["d_model"] * MODEL_CONFIG["d_vocab"]  # embed + unembed
print(f"[model config] {MODEL_CONFIG}")
print(f"[model config] rough attn-param estimate: {n_params_estimate:,} (excludes embed/unembed)")
print(f"[model config] rough embed+unembed estimate: {embed_params_estimate:,}")

## Dataset preparation

Five steps, each its own cell so we can check the output before moving on:

1. **Stream and cache the raw corpus** — pull `CHAR_BUDGET` characters from `DATASET_NAME` (same
   schema the smoke test confirmed: `{"text": ..., "meta": {"pile_set_name": ...}}`), concatenate,
   write to `CORPUS_CACHE` so a re-run doesn't re-download.
2. **Train our own byte-level BPE tokenizer** — GPT-2's algorithm (`ByteLevel` pre-tokenizer +
   decoder, so code/punctuation/non-English text never hits an unknown token), fit to *this*
   corpus rather than reused from `redwood_attn_2l`'s tokenizer, at `VOCAB_SIZE` (2048) — sized
   for this model's width, not GPT-2's own 50257. Cached to `TOKENIZER_CACHE`.
3. **Tokenize and cache token ids** — encode the cached text with our tokenizer, save the id array
   to `TOKENS_CACHE`. Tokenizing 200M characters is slow enough to be worth caching once, not
   redoing per notebook run.
4. **Hold out a fixed evaluation slice** — set aside a chunk of tokens never used for training.
   `stage2c_induction_tinystories.ipynb` found that a fixed held-out batch gives a much
   lower-variance loss reading than sampling fresh minibatches — worth having in place before we
   train, not bolted on after.
5. **Chunk into training sequences** — split the remaining tokens into fixed-length windows of
   `MODEL_CONFIG["n_ctx"]` (2048) for batching.

Each step prints enough to sanity-check itself (corpus size, tokens-per-char, sequence count)
before we commit to the next one.